In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels
import datetime

### Build dataset

In [7]:
CDS_data = pd.read_csv('data/processed/CDS/Weekly_CDS.csv')
CDS_data['Date'] = pd.to_datetime(CDS_data['Date'])
CDS_data.set_index('Date', inplace=True)

Oil_data = pd.read_csv('data/processed/Oil/oil_prices_datastream.csv')
Oil_data['Date'] = pd.to_datetime(Oil_data['Date'])
Oil_data.set_index('Date', inplace=True)

Macro_risk_variables = pd.read_csv('data/processed/Macroeconomic_variables/macro_risk_variables.csv')
Macro_risk_variables['Date'] = pd.to_datetime(Macro_risk_variables['Date'])
Macro_risk_variables.set_index('Date', inplace=True)

VIX = pd.read_csv('data/processed/Macroeconomic_variables/VIXCLS.csv')
VIX['Date'] = pd.to_datetime(VIX['Date'])
VIX.set_index('Date', inplace=True)

OVX = pd.read_csv('data/processed/Macroeconomic_variables/OVXCLS.csv')
OVX['Date'] = pd.to_datetime(OVX['Date'])
OVX.set_index('Date', inplace=True)

merged = CDS_data.copy()
merged = pd.merge(merged, Oil_data, on='Date', how='inner')
merged = pd.merge(merged, Macro_risk_variables, on='Date', how='inner')
merged = pd.merge(merged, VIX, on='Date', how='inner')
merged = pd.merge(merged, OVX, on='Date', how='inner')
#merged.set_index('Date',inplace=True)
merged.head()

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_26573/277222691.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Oil_data['Date'] = pd.to_datetime(Oil_data['Date'])
/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_26573/277222691.py:10: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  Macro_risk_variables['Date'] = pd.to_datetime(Macro_risk_variables['Date'])


,United States,United Kingdom,Japan,Australia,China,France,Germany,India,Italy,Turkey,...,WTI,OPEC_basket,Dubai_Crude,SPX,DXY,UST2Y,UST5Y,UST10Y,VIXCLS,OVXCLS
Date,,,,,,,,,,,,,,,,,,,,,
2007-05-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,62.38,63.15,64.04,1505.85,82.12,4.709,4.590,4.672,12.95,26.41
2007-05-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,64.95,66.01,65.67,1522.75,82.20,4.820,4.730,4.805,12.76,24.71
2007-05-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,64.76,66.57,66.55,1515.73,82.33,4.862,4.800,4.864,13.34,25.81
2007-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,65.09,64.60,65.24,1536.34,82.32,4.976,4.927,4.956,12.78,27.39
2007-06-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,64.76,66.23,63.96,1507.67,82.69,5.004,5.044,5.109,14.84,26.51


## Setting up the regression

### Reshaping data to panel data format

In [10]:
merged = merged[merged.index >= datetime.datetime(2014,1,1)].copy()
merged = merged.reset_index()
oil_exporters = ['Saudi Arabia', 'Abu Dhabi','Dubai','Qatar','Colombia','Mexico','Brazil','Egypt','Malaysia']
controls = ['Indonesia','Philippines','Turkey','Chile','China','South Africa','South Korea','Thailand'] 
# Long format
panel_data = []

for country in oil_exporters + controls:
    temp = merged[['Date', 'Brent', 'OVXCLS', 'VIXCLS', country]].copy()
    temp['Country'] = country
    temp['CDS'] = temp[country]
    temp['OilExporter'] = 1 if country in oil_exporters else 0
    temp = temp[['Date', 'Country', 'CDS', 'Brent', 'OVXCLS', 'VIXCLS', 'OilExporter']]
    panel_data.append(temp)

panel = pd.concat(panel_data, ignore_index=True)
panel = panel.sort_values(by=['Country', 'Date']).reset_index(drop=True)

panel['CDS_ret'] = panel.groupby('Country')['CDS'].pct_change()
panel['Oil_ret'] = panel['Brent'].pct_change()
panel['OVX_chg'] = panel['OVXCLS'].pct_change()
panel['VIX_chg'] = panel['VIXCLS'].pct_change()
panel['OVX x OilExporter'] = panel['OVX_chg'] * panel['OilExporter']


panel = panel.dropna()
panel.head()

,Date,Country,CDS,Brent,OVXCLS,VIXCLS,OilExporter,CDS_ret,Oil_ret,OVX_chg,VIX_chg,OVX x OilExporter
1,2014-01-10,Abu Dhabi,55.85999,106.33,19.47,12.14,1,-0.008168,-0.006911,-0.054854,-0.117733,-0.054854
2,2014-01-17,Abu Dhabi,55.32999,106.89,17.12,12.44,1,-0.009488,0.005267,-0.120699,0.024712,-0.120699
3,2014-01-24,Abu Dhabi,55.34999,107.45,19.46,18.14,1,0.000361,0.005239,0.136682,0.458199,0.136682
4,2014-01-31,Abu Dhabi,56.32999,107.13,20.35,18.41,1,0.017706,-0.002978,0.045735,0.014884,0.045735
5,2014-02-07,Abu Dhabi,56.32999,108.21,19.03,15.29,1,0.000000,0.010081,-0.064865,-0.169473,-0.064865


## Panel regressions

In [11]:
print("\n" + "="*70)
print("MODEL 1: Oil  +  Controls only")
print("="*70)

X1 = sm.add_constant(panel[['VIX_chg','OVX_chg']])
model1 = sm.OLS(panel['CDS_ret'], X1).fit(cov_type='cluster', cov_kwds={'groups': panel['Country']})
print(model1.summary())



MODEL 1: Oil  +  Controls only
                            OLS Regression Results                            
Dep. Variable:                CDS_ret   R-squared:                       0.211
Model:                            OLS   Adj. R-squared:                  0.211
Method:                 Least Squares   F-statistic:                     139.7
Date:                Thu, 29 Jan 2026   Prob (F-statistic):           7.41e-11
Time:                        20:46:06   Log-Likelihood:                 11398.
No. Observations:                9129   AIC:                        -2.279e+04
Df Residuals:                    9126   BIC:                        -2.277e+04
Df Model:                           2                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.000

In [12]:
print("\n" + "="*70)
print("MODEL 2: Oil  +  Dummy")
print("="*70)

X1 = sm.add_constant(panel[['OVX_chg','OVX x OilExporter', 'VIX_chg']])
model1 = sm.OLS(panel['CDS_ret'], X1).fit(cov_type='cluster', cov_kwds={'groups': panel['Country']})
print(model1.summary())



MODEL 2: Oil  +  Dummy
                            OLS Regression Results                            
Dep. Variable:                CDS_ret   R-squared:                       0.211
Model:                            OLS   Adj. R-squared:                  0.211
Method:                 Least Squares   F-statistic:                     110.4
Date:                Thu, 29 Jan 2026   Prob (F-statistic):           6.65e-11
Time:                        20:46:10   Log-Likelihood:                 11401.
No. Observations:                9129   AIC:                        -2.279e+04
Df Residuals:                    9125   BIC:                        -2.276e+04
Df Model:                           3                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const         

## Quantile regressions

In [13]:
print("="*70)
print("DESIGN 1: EXTREME OIL MOVES ONLY (top/bottom 10%)")
print("="*70)

# Define extreme oil moves

for pct in [0.10, 0.05, 0.01, 0.005]:


    oil_qtl = panel['OVX_chg'].quantile(1-pct)

    extreme_panel = panel[(panel['OVX_chg'] > oil_qtl)]

    print(f"Extreme observations: {len(extreme_panel)} ({len(extreme_panel)/len(panel)*100:.1f}%)")

    X = sm.add_constant(extreme_panel[['OVX_chg', 'VIX_chg', 'OVX x OilExporter']])
    model = sm.OLS(extreme_panel['CDS_ret'], X).fit(cov_type='cluster', cov_kwds={'groups': extreme_panel['Country']})
    print(model.summary())

DESIGN 1: EXTREME OIL MOVES ONLY (top/bottom 10%)
Extreme observations: 901 (9.9%)
                            OLS Regression Results                            
Dep. Variable:                CDS_ret   R-squared:                       0.295
Model:                            OLS   Adj. R-squared:                  0.293
Method:                 Least Squares   F-statistic:                     69.48
Date:                Thu, 29 Jan 2026   Prob (F-statistic):           2.15e-09
Time:                        20:46:14   Log-Likelihood:                 672.49
No. Observations:                 901   AIC:                            -1337.
Df Residuals:                     897   BIC:                            -1318.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------